# 대화 히스토리와 질문 재작성

대명사가 들어간 후속 질문을 독립 검색 질문으로 바꾸면 검색 결과가 달라지는지 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 후속 질문에는 ‘그것’, ‘언제’처럼 앞 대화가 있어야 뜻이 정해지는 표현이 남을 수 있습니다. 재작성 함수는 대화 기록과 현재 질문을 입력으로 받아 검색만으로 이해할 수 있는 독립 질문 문자열을 반환합니다. 반환된 문자열로 검색을 실행하며, 원래 사용자 질문과 모델에 보여 줄 최종 질문까지 반드시 바꾸는 것은 아닙니다.

**다른 상황에 적용.** 쇼핑 상담에서 ‘그 제품은 반품돼?’를 ‘무선 키보드 K2는 개봉 후 반품할 수 있나요?’로 바꾸는 데도 같은 원리를 적용할 수 있습니다. 앞 대화의 대상과 현재 질문의 조건을 함께 보존해야 합니다.

**비교와 경계.** 질문 재작성은 빠진 대상을 복원하는 처리이고 요약은 긴 대화의 정보를 줄이는 처리입니다. 재작성 결과에 없던 날짜나 상품명을 보태면 검색은 선명해져도 사용자의 뜻이 달라질 수 있으므로 원문과 재작성문을 함께 기록합니다.


## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

**이 셀의 코드 읽기.** `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

**이 셀의 코드 읽기.** `search(question: str, k: int = 3)`, `ask(prompt: str)`, `show_hits(hits: list[tuple[dict[str, Any], float]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다.


In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 히스토리가 없는 검색

대명사가 남은 후속 질문을 그대로 검색합니다. 출력에서 LIB-004의 기준선 순위를 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 대명사만 있는 후속 질문을 그대로 검색해 기준선을 만듭니다.
history=["사용자: 일반 도서 대출 기간은?","도우미: 14일이며 한 번 연장할 수 있습니다."]
follow_up="그거 말고 전자책은?"
before=search(follow_up)
print("Before:",show_hits(before))

Before: [('ACC-002', 0.381), ('LIB-004', 0.32), ('LIB-003', 0.264)]


## 모델로 독립 질문 만들기

대화 기록과 후속 질문을 모델에 입력해 독립 질문을 만듭니다. 복원된 대상과 재작성 뒤 문서 순위를 비교합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
# 대화 기록에서 전자책이라는 대상을 복원해 검색 질문을 만듭니다.
prompt=f"""대화 기록을 참고해 마지막 질문을 한 문장의 독립 검색 질문으로 바꾸세요.
대화 기록: {history}
마지막 질문: {follow_up}
질문만 출력하세요."""
rewritten,tokens,latency=ask(prompt)
after=search(rewritten)
print("재작성:",rewritten)
print("After:",show_hits(after),"토큰:",tokens,"ms:",latency)

재작성: 전자책 대출 기간은 어떻게 되나요?
After: [('LIB-004', 0.584), ('LIB-001', 0.521), ('ACC-002', 0.379)] 토큰: 91 ms: 1329.1


## 검색 결과로 답하기

재작성 질문으로 찾은 원문만 모델에 전달합니다. 답의 대출 기간과 인용 문서 ID를 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [8]:
# 검색된 원문과 문서 ID만 답변 근거로 전달합니다.
context="\n".join(f"[{d['doc_id']}] {d['text']}" for d,_ in after)
answer,tokens,latency=ask(f"근거만 사용해 답하고 [문서ID]를 쓰세요.\n{context}\n질문:{rewritten}")
print(answer)
print("토큰·ms:",tokens,latency)

전자책 대출 기간은 7일입니다. [LIB-004]
토큰·ms: 145 832.7
